In [1]:
import pandas as pd
import numpy as np

In [2]:
balls = pd.read_parquet("../ml-service/data/processed/v4_beta/clean_deliveries.parquet")
matches = pd.read_parquet("../ml-service/data/processed/v4_beta/clean_matches.parquet")

In [3]:
balls.head(2)

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs,total_balls
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,Sourav Ganguly,Brendon McCullum,Praveen Kumar,0,0,0,0.0,1.0,0.0,Not Out,2008-04-18,1.0,1
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,0,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0,2


In [4]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs', 'total_balls'],
      dtype='object')

In [5]:
balls.groupby(["matchId", "inning", "over", "total_balls"]).size().value_counts()

1    273853
Name: count, dtype: int64

In [6]:
balls.shape

(273853, 19)

In [7]:
balls = balls.sort_values(["matchId", "inning", "over", "total_balls"]).reset_index(
    drop=True
)

In [8]:
balls.groupby(["matchId", "inning"])["matchId"].nunique().value_counts()

matchId
1    2284
Name: count, dtype: int64

In [9]:
balls['isWide'].value_counts()

isWide
0    264911
1      8201
2       355
5       325
3        54
4         7
Name: count, dtype: int64

In [10]:
balls['player_dismissed'][balls['isWide']>1].unique()

array(['Not Out'], dtype=object)

In [11]:
mask = balls["isNoBall"] > 1

balls.loc[mask, "batsman_runs"] += balls.loc[mask, "isNoBall"] - 1
balls.loc[mask, "isNoBall"] = 1

In [12]:
balls["total_balls"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [13]:
balls["repeat"] = np.where(balls["isWide"] > 0, balls["isWide"], 1)

balls = balls.loc[balls.index.repeat(balls["repeat"])].copy()

balls.loc[balls["isWide"] > 0, "isWide"] = 1

balls.drop(columns=["repeat"], inplace=True)

In [14]:
balls.shape

(275637, 19)

In [15]:
balls["is_legal"] = ((balls["isWide"] == 0) & (balls["isNoBall"] == 0)).astype(int)

running_legal_count = balls.groupby(["matchId", "inning", "over"])["is_legal"].cumsum()

balls["legal_ball"] = (
    running_legal_count.groupby([balls["matchId"], balls["inning"], balls["over"]])
    .shift(1)
    .fillna(0) + 1
)

In [16]:
balls = balls.reset_index(drop=True)

In [17]:
balls.shape

(275637, 21)

In [18]:
balls = balls[balls["legal_ball"] <= 6]

In [19]:
balls.shape

(275601, 21)

In [20]:
balls["legal_ball"].unique()

array([1., 2., 3., 4., 5., 6.])

In [21]:
balls.groupby(["matchId", "inning", "over"])["is_legal"].sum().max()

np.int64(6)

In [22]:
balls.head(2)

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs,total_balls,is_legal,legal_ball
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,Sourav Ganguly,Brendon McCullum,Praveen Kumar,0,...,0,0.0,1.0,0.0,Not Out,2008-04-18,1.0,1,1,1.0
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,...,0,0.0,0.0,0.0,Not Out,2008-04-18,0.0,2,1,2.0


In [23]:
balls["legal_ball_1"] = (balls["isWide"] == 0) & (balls["isNoBall"] == 0)

TOTAL_BALLS = 120

balls["balls_bowled"] = (
    balls.groupby(["matchId", "inning"])["legal_ball_1"]
    .cumsum()
    .groupby([balls["matchId"], balls["inning"]])
    .shift(fill_value=0)
)

balls["balls_remaining"] = TOTAL_BALLS - balls["balls_bowled"]

In [24]:
balls['balls_remaining'].unique()

array([120, 119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108,
       107, 106, 105, 104, 103, 102, 101, 100,  99,  98,  97,  96,  95,
        94,  93,  92,  91,  90,  89,  88,  87,  86,  85,  84,  83,  82,
        81,  80,  79,  78,  77,  76,  75,  74,  73,  72,  71,  70,  69,
        68,  67,  66,  65,  64,  63,  62,  61,  60,  59,  58,  57,  56,
        55,  54,  53,  52,  51,  50,  49,  48,  47,  46,  45,  44,  43,
        42,  41,  40,  39,  38,  37,  36,  35,  34,  33,  32,  31,  30,
        29,  28,  27,  26,  25,  24,  23,  22,  21,  20,  19,  18,  17,
        16,  15,  14,  13,  12,  11,  10,   9,   8,   7,   6,   5,   4,
         3,   2,   1])

In [25]:
balls.head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,Penalty,player_dismissed,date,total_runs,total_balls,is_legal,legal_ball,legal_ball_1,balls_bowled,balls_remaining
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,Sourav Ganguly,Brendon McCullum,Praveen Kumar,0,...,0.0,Not Out,2008-04-18,1.0,1,1,1.0,True,0,120
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,...,0.0,Not Out,2008-04-18,0.0,2,1,2.0,True,1,119
2,335982,0,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,...,0.0,Not Out,2008-04-18,1.0,3,0,3.0,False,2,118
3,335982,0,0,4,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,...,0.0,Not Out,2008-04-18,0.0,4,1,3.0,True,2,118
4,335982,0,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0,...,0.0,Not Out,2008-04-18,0.0,5,1,4.0,True,3,117


In [26]:
balls.groupby(["matchId", "inning"])["legal_ball_1"].sum().max()

np.int64(120)

In [27]:
balls = balls.reset_index(drop=True)

In [28]:
balls["balls_remaining"].unique()

array([120, 119, 118, 117, 116, 115, 114, 113, 112, 111, 110, 109, 108,
       107, 106, 105, 104, 103, 102, 101, 100,  99,  98,  97,  96,  95,
        94,  93,  92,  91,  90,  89,  88,  87,  86,  85,  84,  83,  82,
        81,  80,  79,  78,  77,  76,  75,  74,  73,  72,  71,  70,  69,
        68,  67,  66,  65,  64,  63,  62,  61,  60,  59,  58,  57,  56,
        55,  54,  53,  52,  51,  50,  49,  48,  47,  46,  45,  44,  43,
        42,  41,  40,  39,  38,  37,  36,  35,  34,  33,  32,  31,  30,
        29,  28,  27,  26,  25,  24,  23,  22,  21,  20,  19,  18,  17,
        16,  15,  14,  13,  12,  11,  10,   9,   8,   7,   6,   5,   4,
         3,   2,   1])

In [29]:
balls["over_number"] = balls["over"].astype(int) + 1

balls["phase_pp"] = (balls["over_number"] <= 6).astype(int)
balls["phase_middle"] = ((balls["over_number"] > 6) & (balls["over_number"] <= 15)).astype(int)
balls["phase_death"] = (balls["over_number"] > 15).astype(int)

In [30]:
balls[balls['phase_middle']==1].head(2)

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,total_balls,is_legal,legal_ball,legal_ball_1,balls_bowled,balls_remaining,over_number,phase_pp,phase_middle,phase_death
42,335982,0,6,1,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Ricky Ponting,Andrew Noffke,1,...,1,1,1.0,True,36,84,7,0,1,0
43,335982,0,6,2,Kolkata Knight Riders,Royal Challengers Bangalore,Ricky Ponting,Brendon McCullum,Andrew Noffke,1,...,2,1,2.0,True,37,83,7,0,1,0


In [31]:
balls.isna().sum()

matchId             0
inning              0
over                0
ball                0
batting_team        0
bowling_team        0
batsman             0
non_striker         0
bowler              0
batsman_runs        0
isWide              0
isNoBall            0
Byes                0
LegByes             0
Penalty             0
player_dismissed    0
date                0
total_runs          0
total_balls         0
is_legal            0
legal_ball          0
legal_ball_1        0
balls_bowled        0
balls_remaining     0
over_number         0
phase_pp            0
phase_middle        0
phase_death         0
dtype: int64

In [32]:
balls["total_runs"] = (
    balls["batsman_runs"]
    + balls["isWide"]
    + balls["isNoBall"]
    + balls["Byes"]
    + balls["LegByes"]
    + balls["Penalty"]
)
balls["current_score"] = balls.groupby(["matchId", "inning"])["total_runs"].cumsum()
# balls.drop(columns=['total_runs'], inplace=True)

In [33]:
balls[["over", "legal_ball", "current_score", "isWide", "isNoBall"]].head(10)

,over,legal_ball,current_score,isWide,isNoBall
0,0,1.0,1.0,0,0
1,0,2.0,1.0,0,0
2,0,3.0,2.0,1,0
3,0,3.0,2.0,0,0
4,0,4.0,2.0,0,0
5,0,5.0,2.0,0,0
6,0,6.0,3.0,0,0
7,1,1.0,3.0,0,0
8,1,2.0,7.0,0,0
9,1,3.0,11.0,0,0


In [34]:
balls["current_score"].unique().max()

np.float64(287.0)

In [35]:
balls.loc[balls["Penalty"] == 5, "batsman_runs"] = 5

In [36]:
balls["batsman_runs"] = balls["batsman_runs"] + balls["Byes"] + balls["LegByes"]
balls["batsman_runs"].unique()

array([1., 0., 4., 6., 2., 5., 3.])

In [37]:
balls = balls.reset_index(drop=True)

In [38]:
len(balls["player_dismissed"].unique())

648

In [39]:
balls["is_wicket"] = (balls["player_dismissed"] != "Not Out").astype(int)
balls["wickets_fallen"] = balls.groupby(["matchId", "inning"])["is_wicket"].cumsum()

In [40]:
balls["wickets_fallen"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [41]:
balls = balls.reset_index(drop=True)

In [42]:
first_innings_score = (
    balls[balls["inning"] == 0].groupby("matchId")["current_score"].max()
)
balls["target"] = balls["matchId"].map(first_innings_score)
balls.loc[balls["inning"] == 1, "target"] = balls["target"] + 1
balls.loc[balls["inning"] == 0, "target"] = 0

In [43]:
balls[balls["target"] > 0].head(2)

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,balls_bowled,balls_remaining,over_number,phase_pp,phase_middle,phase_death,current_score,is_wicket,wickets_fallen,target
129,335982,1,0,1,Royal Challengers Bangalore,Kolkata Knight Riders,Rahul Dravid,Wasim Jaffer,Ashok Dinda,1.0,...,0,120,1,1,0,0,1.0,0,0,223.0
130,335982,1,0,2,Royal Challengers Bangalore,Kolkata Knight Riders,Wasim Jaffer,Rahul Dravid,Ashok Dinda,0.0,...,1,119,1,1,0,0,2.0,0,0,223.0


In [44]:
balls["target"].unique().max()

np.float64(288.0)

In [45]:
over_runs = (
    balls.groupby(["matchId", "inning", "over_number"])["total_runs"]
    .sum()
    .reset_index(name="over_runs")
)
over_runs["last_over_runs"] = over_runs.groupby(["matchId", "inning"])[
    "over_runs"
].shift(1)
balls = balls.merge(
    over_runs[["matchId", "inning", "over_number", "last_over_runs"]],
    on=["matchId", "inning", "over_number"],
    how="left",
)
balls["last_over_runs"] = balls["last_over_runs"].fillna(0)
balls["last_over_runs"] = balls["last_over_runs"].astype(int)

In [46]:
balls[(balls['last_over_runs']>0)&(balls['over']==0)]

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,...,balls_remaining,over_number,phase_pp,phase_middle,phase_death,current_score,is_wicket,wickets_fallen,target,last_over_runs


In [47]:
balls["total_balls"] = balls.groupby(["matchId", "inning", "over"]).cumcount() + 1

In [48]:
balls["total_balls"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])

In [49]:
balls.groupby(
    ["matchId", "inning", "over", "legal_ball", "total_balls"]
).size().value_counts()

1    275601
Name: count, dtype: int64

In [50]:
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "batsman_runs",
] = 3
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "total_runs",
] = 4
balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] == 5),
    "current_score",
] = 181
to_drop = balls.loc[
    (balls["matchId"] == 1254073)
    & (balls["inning"] == 1)
    & (balls["over"] == 16)
    & (balls["total_balls"] > 5)
].index

balls = balls.drop(to_drop)

In [51]:
balls.shape

(275598, 33)

In [52]:
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "batsman_runs",
] = 2
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "total_runs",
] = 3
balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] == 5),
    "current_score",
] = 111
to_drop = balls.loc[
    (balls["matchId"] == 1178398)
    & (balls["inning"] == 1)
    & (balls["over"] == 17)
    & (balls["total_balls"] > 5)
].index

balls = balls.drop(to_drop)

In [53]:
balls.shape

(275596, 33)

In [54]:
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "batsman_runs",
] = 6
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "total_runs",
] = 6
balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] == 4),
    "current_score",
] = 131
to_drop = balls.loc[
    (balls["matchId"] == 729309)
    & (balls["inning"] == 1)
    & (balls["over"] == 18)
    & (balls["total_balls"] > 4)
].index

balls = balls.drop(to_drop)

In [55]:
balls.shape

(275595, 33)

In [56]:
balls = balls.sort_values(["matchId", "inning", "over", "total_balls"]).reset_index(
    drop=True
)

In [57]:
balls["current_score"] = balls.groupby(["matchId", "inning"])["total_runs"].cumsum()
balls["wickets_fallen"] = balls.groupby(["matchId", "inning"])["is_wicket"].cumsum()

In [58]:
mask = (balls["isNoBall"] == 1) & (balls["player_dismissed"] != "Not Out")
balls.loc[mask, "isWide"] = 1
balls.loc[mask, "isNoBall"] = 0
len(balls[(balls["isNoBall"] == 1) & (balls["player_dismissed"] != "Not Out")])

0

In [59]:
balls["batsman_runs_target"] = balls["batsman_runs"].astype(int)
balls["isWide_target"] = balls["isWide"].astype(int)
balls["isNoBall_target"] = balls["isNoBall"].astype(int)
balls["is_wicket_target"] = balls["is_wicket"].astype(int)

In [60]:
balls["is_boundary"] = balls["batsman_runs"].isin([4, 6]).astype(int)


def compute_balls_since_boundary(x):
    groups = x.cumsum()
    result = x.groupby(groups).cumcount()
    result[groups == 0] = range(1, (groups == 0).sum()+1)

    return result


balls["balls_since_boundary"] = balls.groupby(["matchId", "inning"])[
    "is_boundary"
].transform(compute_balls_since_boundary)

balls["balls_since_boundary"] = (
    balls.groupby(["matchId", "inning"])["balls_since_boundary"]
    .shift(1)
    .fillna(0)
)

balls['balls_since_boundary'] = balls['balls_since_boundary'].astype(int)

In [61]:
balls[['batsman_runs','balls_since_boundary']].head(10)

,batsman_runs,balls_since_boundary
0,1.0,0
1,0.0,1
2,0.0,2
3,0.0,3
4,0.0,4
5,0.0,5
6,1.0,6
7,0.0,7
8,4.0,8
9,4.0,0


In [62]:
balls["balls_since_boundary"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [63]:
len(
    balls[
        (balls["balls_since_boundary"] > 0)
        & (balls["over"] == 0)
        & (balls["total_balls"] == 1)
    ]
)

0

In [ ]:
for col in ["batsman_runs", "isWide", "isNoBall", "is_wicket","total_runs"]:
    balls[f"prev_{col}"] = balls.groupby(["matchId", "inning"])[col].shift(1).fillna(0)

balls["score_before"] = balls.groupby(["matchId", "inning"])['current_score'].shift(1).fillna(0)
balls["wickets_before"] = balls.groupby(["matchId", "inning"])['wickets_fallen'].shift(1).fillna(0)

In [65]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.6f}'.format)

In [66]:
balls[['inning','wickets_before','wickets_fallen','player_dismissed']].head()

,inning,wickets_before,wickets_fallen,player_dismissed
0,0,0.000000,0,Not Out
1,0,0.000000,0,Not Out
2,0,0.000000,0,Not Out
3,0,0.000000,0,Not Out
4,0,0.000000,0,Not Out


In [67]:
balls['wickets_before'].unique()

array([0., 1., 2., 3., 4., 5., 6., 7., 8., 9.])

In [68]:
balls["percentage_target_achieved"] = np.where(
    balls["inning"] == 0, 0.0, balls["score_before"] / balls["target"]
)
balls["percentage_target_achieved"] = (
    balls["percentage_target_achieved"].replace([np.inf, -np.inf], 0).fillna(0)
)

In [69]:
balls["percentage_target_achieved"][balls["inning"] == 1].head(2)

129   0.000000
130   0.004484
Name: percentage_target_achieved, dtype: float64

In [70]:
balls['percentage_target_achieved'].unique().max()

np.float64(0.9961832061068703)

In [71]:
balls = balls.merge(matches[["matchId", "venue"]], on="matchId", how="left")

In [72]:
balls["venue"].head(240)

0      Chinnaswamy Stadium, Bengaluru
1      Chinnaswamy Stadium, Bengaluru
2      Chinnaswamy Stadium, Bengaluru
3      Chinnaswamy Stadium, Bengaluru
4      Chinnaswamy Stadium, Bengaluru
5      Chinnaswamy Stadium, Bengaluru
6      Chinnaswamy Stadium, Bengaluru
7      Chinnaswamy Stadium, Bengaluru
8      Chinnaswamy Stadium, Bengaluru
9      Chinnaswamy Stadium, Bengaluru
10     Chinnaswamy Stadium, Bengaluru
11     Chinnaswamy Stadium, Bengaluru
12     Chinnaswamy Stadium, Bengaluru
13     Chinnaswamy Stadium, Bengaluru
14     Chinnaswamy Stadium, Bengaluru
15     Chinnaswamy Stadium, Bengaluru
16     Chinnaswamy Stadium, Bengaluru
17     Chinnaswamy Stadium, Bengaluru
18     Chinnaswamy Stadium, Bengaluru
19     Chinnaswamy Stadium, Bengaluru
20     Chinnaswamy Stadium, Bengaluru
21     Chinnaswamy Stadium, Bengaluru
22     Chinnaswamy Stadium, Bengaluru
23     Chinnaswamy Stadium, Bengaluru
24     Chinnaswamy Stadium, Bengaluru
25     Chinnaswamy Stadium, Bengaluru
26     Chinn

In [73]:
balls["balls_bowled"] = (
    balls.groupby(["matchId", "inning"])["legal_ball_1"]
    .cumsum()
    .groupby([balls["matchId"], balls["inning"]])
    .shift(fill_value=0)
)

balls["balls_remaining"] = TOTAL_BALLS - balls["balls_bowled"]

balls["overs_bowled"] = balls["balls_bowled"] / 6

balls["current_run_rate"] = np.where(
    balls["balls_bowled"] > 0, balls["score_before"] / balls["overs_bowled"], 0
)

In [74]:
balls['balls_bowled'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119])

In [75]:
balls.head(3)

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs,total_balls,is_legal,legal_ball,legal_ball_1,balls_bowled,balls_remaining,over_number,phase_pp,phase_middle,phase_death,current_score,is_wicket,wickets_fallen,target,last_over_runs,batsman_runs_target,isWide_target,isNoBall_target,is_wicket_target,is_boundary,balls_since_boundary,prev_batsman_runs,prev_isWide,prev_isNoBall,prev_is_wicket,prev_total_runs,score_before,wickets_before,prev_player_dismissed,percentage_target_achieved,venue,overs_bowled,current_run_rate
0,335982,0,0,1,Kolkata Knight Riders,Royal Challengers Bangalore,Sourav Ganguly,Brendon McCullum,Praveen Kumar,1.000000,0,0,0.000000,1.000000,0.000000,Not Out,2008-04-18,1.000000,1,1,1.000000,True,0,120,1,1,0,0,1.000000,0,0,0.000000,0,1,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Not Out,0.000000,"Chinnaswamy Stadium, Bengaluru",0.000000,0.000000
1,335982,0,0,2,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0.000000,0,0,0.000000,0.000000,0.000000,Not Out,2008-04-18,0.000000,2,1,2.000000,True,1,119,1,1,0,0,1.000000,0,0,0.000000,0,0,0,0,0,0,1,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,Not Out,0.000000,"Chinnaswamy Stadium, Bengaluru",0.166667,6.000000
2,335982,0,0,3,Kolkata Knight Riders,Royal Challengers Bangalore,Brendon McCullum,Sourav Ganguly,Praveen Kumar,0.000000,1,0,0.000000,0.000000,0.000000,Not Out,2008-04-18,1.000000,3,0,3.000000,False,2,118,1,1,0,0,2.000000,0,0,0.000000,0,0,1,0,0,0,2,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,Not Out,0.000000,"Chinnaswamy Stadium, Bengaluru",0.333333,3.000000


In [76]:
balls[['over','legal_ball','current_score','score_before','current_run_rate']].head(135)

,over,legal_ball,current_score,score_before,current_run_rate
0,0,1.000000,1.000000,0.000000,0.000000
1,0,2.000000,1.000000,1.000000,6.000000
2,0,3.000000,2.000000,1.000000,3.000000
3,0,3.000000,2.000000,2.000000,6.000000
4,0,4.000000,2.000000,2.000000,4.000000
5,0,5.000000,2.000000,2.000000,3.000000
6,0,6.000000,3.000000,2.000000,2.400000
7,1,1.000000,3.000000,3.000000,3.000000
8,1,2.000000,7.000000,3.000000,2.571429
9,1,3.000000,11.000000,7.000000,5.250000


In [77]:
balls["runs_required"] = balls["target"] - balls["score_before"]

balls["required_run_rate"] = np.where(
    (balls["balls_remaining"] > 0) & (balls["runs_required"] > 0),
    balls["runs_required"] * 6 / balls["balls_remaining"],
    0
)

balls["required_run_rate"] = balls["required_run_rate"].replace([np.inf, -np.inf], 0)
balls["required_run_rate"] = balls["required_run_rate"].fillna(0)

balls.loc[balls["inning"] == 0, "required_run_rate"] = 0
balls.loc[balls["runs_required"] <= 0, "required_run_rate"] = 0

In [78]:
balls[['over','legal_ball','current_score','score_before','current_run_rate','required_run_rate']][balls['inning']==1].head(15)

,over,legal_ball,current_score,score_before,current_run_rate,required_run_rate
129,0,1.000000,1.000000,0.000000,0.000000,11.150000
130,0,2.000000,2.000000,1.000000,6.000000,11.193277
131,0,2.000000,2.000000,2.000000,12.000000,11.142857
132,0,3.000000,3.000000,2.000000,6.000000,11.237288
133,0,4.000000,4.000000,3.000000,6.000000,11.282051
134,0,5.000000,4.000000,4.000000,6.000000,11.327586
135,0,6.000000,4.000000,4.000000,4.800000,11.426087
136,1,1.000000,4.000000,4.000000,4.000000,11.526316
137,1,2.000000,4.000000,4.000000,3.428571,11.628319
138,1,3.000000,8.000000,4.000000,3.000000,11.732143


In [79]:
balls[["current_run_rate", "required_run_rate"]].describe()

,current_run_rate,required_run_rate
count,275595.000000,275595.000000
mean,7.588302,5.249580
std,2.521264,10.590587
min,0.000000,0.000000
25%,6.344828,0.000000
50%,7.630435,0.000000
75%,8.903226,8.973913
max,66.000000,792.000000


In [80]:
pd.set_option("display.max_columns", None)
balls[balls["current_run_rate"] > 60].head()

,matchId,inning,over,ball,batting_team,bowling_team,batsman,non_striker,bowler,batsman_runs,isWide,isNoBall,Byes,LegByes,Penalty,player_dismissed,date,total_runs,total_balls,is_legal,legal_ball,legal_ball_1,balls_bowled,balls_remaining,over_number,phase_pp,phase_middle,phase_death,current_score,is_wicket,wickets_fallen,target,last_over_runs,batsman_runs_target,isWide_target,isNoBall_target,is_wicket_target,is_boundary,balls_since_boundary,prev_batsman_runs,prev_isWide,prev_isNoBall,prev_is_wicket,prev_total_runs,score_before,wickets_before,prev_player_dismissed,percentage_target_achieved,venue,overs_bowled,current_run_rate,runs_required,required_run_rate
46919,501222,0,0,5,Kolkata Knight Riders,Royal Challengers Bangalore,Jacques Kallis,Brad Haddin,Zaheer Khan,0.000000,0,0,0.000000,0.000000,0.000000,Not Out,2011-04-22,0.000000,9,1,2.000000,True,1,119,1,1,0,0,11.000000,0,0,0.000000,0,0,0,0,0,0,7,0.000000,1.000000,0.000000,0.000000,1.000000,11.000000,0.000000,Not Out,0.000000,"Eden Gardens, Kolkata",0.166667,66.000000,-11.000000,0.000000


In [81]:
key_cols = ["matchId", "inning", "over", "total_balls"]

multi_batters = (
    balls.groupby(key_cols)["batsman"].nunique().reset_index(name="batter_count")
)

bad_balls_batter = multi_batters[multi_batters["batter_count"] > 1]

print(bad_balls_batter.shape)
bad_balls_batter.head()

(0, 5)


,matchId,inning,over,total_balls,batter_count


In [82]:
multi_bowlers = (
    balls.groupby(key_cols)["bowler"].nunique().reset_index(name="bowler_count")
)

bad_balls_bowler = multi_bowlers[multi_bowlers["bowler_count"] > 1]

print(bad_balls_bowler.shape)
bad_balls_bowler.head()

(0, 5)


,matchId,inning,over,total_balls,bowler_count


In [83]:
balls.index.is_unique

True

In [84]:
len(balls[(balls["isWide"] == 1) & (balls["player_dismissed"] != "Not Out")])

60

In [85]:
balls["over"] = balls["over"] / 20
np.set_printoptions(suppress=True)
balls["over"].unique()

array([0.  , 0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 ,
       0.55, 0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [86]:
season_map = matches.set_index("matchId")["season"]
balls["season"] = balls["matchId"].map(season_map)

In [87]:
# create cyclic features
balls["sin_ball"] = np.sin(2 * np.pi * balls["legal_ball"] / 6)
balls["cos_ball"] = np.cos(2 * np.pi * balls["legal_ball"] / 6)

In [88]:
balls.columns

Index(['matchId', 'inning', 'over', 'ball', 'batting_team', 'bowling_team',
       'batsman', 'non_striker', 'bowler', 'batsman_runs', 'isWide',
       'isNoBall', 'Byes', 'LegByes', 'Penalty', 'player_dismissed', 'date',
       'total_runs', 'total_balls', 'is_legal', 'legal_ball', 'legal_ball_1',
       'balls_bowled', 'balls_remaining', 'over_number', 'phase_pp',
       'phase_middle', 'phase_death', 'current_score', 'is_wicket',
       'wickets_fallen', 'target', 'last_over_runs', 'batsman_runs_target',
       'isWide_target', 'isNoBall_target', 'is_wicket_target', 'is_boundary',
       'balls_since_boundary', 'prev_batsman_runs', 'prev_isWide',
       'prev_isNoBall', 'prev_is_wicket', 'prev_total_runs', 'score_before',
       'wickets_before', 'prev_player_dismissed', 'percentage_target_achieved',
       'venue', 'overs_bowled', 'current_run_rate', 'runs_required',
       'required_run_rate', 'season', 'sin_ball', 'cos_ball'],
      dtype='object')

In [89]:
balls.drop(
    columns=[
        "ball",
        "batting_team",
        "bowling_team",
        "Byes",
        "LegByes",
        "Penalty",
        "over_number",
        "is_legal",
        "legal_ball_1",
        "is_boundary",
        "legal_ball",
        "balls_bowled",
        "overs_bowled",
        "runs_required",
        "date",
        "over_number",
        "batsman_runs",
        "isWide",
        "isNoBall",
        "current_score", 
        "is_wicket",
        "wickets_fallen",
        "total_runs",
        "player_dismissed"
    ],
    inplace=True,
)

In [91]:
# Check for data integrity
assert (balls['wickets_before'] >= 0).all() and (balls['wickets_before'] <= 10).all()
assert (balls['balls_remaining'] >= 0).all() and (balls['balls_remaining'] <= 120).all()
assert balls.isna().sum().sum() == 0, "NaN found in final dataset"


# Check for no NaNs in critical features
assert balls.isna().sum().sum() == 0

# Check target distribution
print(f"Wicket rate: {balls['is_wicket_target'].mean():.2%}")
print(f"Wide rate: {balls['isWide_target'].mean():.2%}")

Wicket rate: 4.93%
Wide rate: 3.88%


In [92]:
balls.columns

Index(['matchId', 'inning', 'over', 'batsman', 'non_striker', 'bowler',
       'total_balls', 'balls_remaining', 'phase_pp', 'phase_middle',
       'phase_death', 'target', 'last_over_runs', 'batsman_runs_target',
       'isWide_target', 'isNoBall_target', 'is_wicket_target',
       'balls_since_boundary', 'prev_batsman_runs', 'prev_isWide',
       'prev_isNoBall', 'prev_is_wicket', 'prev_total_runs', 'score_before',
       'wickets_before', 'prev_player_dismissed', 'percentage_target_achieved',
       'venue', 'current_run_rate', 'required_run_rate', 'season', 'sin_ball',
       'cos_ball'],
      dtype='object')

In [93]:
balls[(balls['balls_remaining']!=120)&(balls['over']==0)&(balls['total_balls']==1)]

,matchId,inning,over,batsman,non_striker,bowler,total_balls,balls_remaining,phase_pp,phase_middle,phase_death,target,last_over_runs,batsman_runs_target,isWide_target,isNoBall_target,is_wicket_target,balls_since_boundary,prev_batsman_runs,prev_isWide,prev_isNoBall,prev_is_wicket,prev_total_runs,score_before,wickets_before,prev_player_dismissed,percentage_target_achieved,venue,current_run_rate,required_run_rate,season,sin_ball,cos_ball


In [94]:
balls["prev_batsman_runs"] = balls["prev_batsman_runs"] / 6
balls["prev_total_runs"] = balls["prev_total_runs"] / 6
balls["balls_remaining"] = balls["balls_remaining"] / 120
balls["wickets_before"] = balls["wickets_before"] / 10
balls["balls_since_boundary"] = balls["balls_since_boundary"] / 120

In [95]:
balls["score_before"] = balls["score_before"] / 200
balls["target"] = balls["target"] / 200
balls["last_over_runs"] = balls["last_over_runs"] / 36
balls["total_balls"] = balls["total_balls"] / 10

In [96]:
balls["season"] = (balls["season"] - 2007) / 20

In [97]:
balls["current_run_rate"] = balls["current_run_rate"] / 36
balls["required_run_rate"] = balls["required_run_rate"] / 36
balls["required_run_rate"] = balls["required_run_rate"].clip(upper=2)

In [98]:
balls["required_run_rate"].unique().max()

np.float64(2.0)

In [99]:
# Verify target distributions
print("batsman_runs_target:", balls["batsman_runs_target"].value_counts().sort_index())
print("isWide_target:", balls["isWide_target"].value_counts())
print("isNoBall_target:", balls["isNoBall_target"].value_counts())
print("is_wicket_target:", balls["is_wicket_target"].value_counts())

# Verify no NaN in targets
print(
    "NaN count:",
    balls[
        ["batsman_runs_target", "isWide_target", "isNoBall_target", "is_wicket_target"]
    ]
    .isna()
    .sum(),
)

batsman_runs_target: batsman_runs_target
0    105449
1    105665
2     17377
3       823
4     32115
5        73
6     14093
Name: count, dtype: int64
isWide_target: isWide_target
0    264895
1     10700
Name: count, dtype: int64
isNoBall_target: isNoBall_target
0    274471
1      1124
Name: count, dtype: int64
is_wicket_target: is_wicket_target
0    262015
1     13580
Name: count, dtype: int64
NaN count: batsman_runs_target    0
isWide_target          0
isNoBall_target        0
is_wicket_target       0
dtype: int64


In [100]:
balls.columns

Index(['matchId', 'inning', 'over', 'batsman', 'non_striker', 'bowler',
       'total_balls', 'balls_remaining', 'phase_pp', 'phase_middle',
       'phase_death', 'target', 'last_over_runs', 'batsman_runs_target',
       'isWide_target', 'isNoBall_target', 'is_wicket_target',
       'balls_since_boundary', 'prev_batsman_runs', 'prev_isWide',
       'prev_isNoBall', 'prev_is_wicket', 'prev_total_runs', 'score_before',
       'wickets_before', 'prev_player_dismissed', 'percentage_target_achieved',
       'venue', 'current_run_rate', 'required_run_rate', 'season', 'sin_ball',
       'cos_ball'],
      dtype='object')

In [101]:
balls.to_csv("data2.csv")